# Verify rung 0 yourself

This notebook re-derives every promoted claim of the rung-0 replicate ceiling from the
committed files in this repository. **Every cell computes what it checks in plain
sight**: standard-library hashing, `pandas` reads of the committed tables, and explicit
arithmetic — nothing is imported from this repository's own code, so there is no wrapper
to trust. Each cell prints the recorded value next to the one recomputed in front of
you and stops with an error if they disagree. Run all cells (Run → Run All Cells;
about a minute on a laptop).

To run it: from the repository root, `uv sync --extra dev`, then
`uv run jupyter lab docs/tasks/rung0-replicate-ceiling/verify.ipynb`.

**What is not checkable locally, stated rather than hidden:** the gene and drug panel
files live on the Alpine cluster and are pinned by checksum in the provenance record, so
the declared panel size (14,121 genes) is a recorded input property here, not a
recomputable one; and the 1,026 data shards sit on cluster scratch, so their integrity
reduces locally to the committed shard manifest hashing to the promoted record's data
version (checked in section 2).

In [ ]:
import hashlib
import json
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert repo is not None, 'run this notebook from inside the repository (its own folder works)'
task = repo / 'docs' / 'tasks' / 'rung0-replicate-ceiling'
results = repo / 'results' / 'rung0-replicate-ceiling'
print('repository:', repo)

## 1. The promoted number cannot have been edited

The promoted table's checksum was written into its provenance record at promotion time.
Hash the file as it sits in the repository and compare — same for the cluster job log —
and confirm the task-folder copy is byte-identical to the promoted copy.

In [ ]:
record = json.loads((results / 'rung0_delta_reproducibility.provenance.json').read_text())

result_sha = hashlib.sha256((repo / record['result']).read_bytes()).hexdigest()
print('recorded result_sha256 :', record['result_sha256'])
print('recomputed sha256      :', result_sha)
assert result_sha == record['result_sha256']

log_sha = hashlib.sha256((repo / record['log']).read_bytes()).hexdigest()
print('recorded log_sha256    :', record['log_sha256'])
print('recomputed sha256      :', log_sha)
assert log_sha == record['log_sha256']

assert (task / 'rung0_delta_reproducibility.csv').read_bytes() == (repo / record['result']).read_bytes()
print('task-folder copy is byte-identical to the promoted copy')

## 2. The input data are pinned

The 1,026 downloaded shards are described by a committed manifest — path, size, and
checksum per shard, one line each. The checksum of that manifest file is the content
hash the tranche record carries, and the promoted record pins the same value as its
data version. Hash it yourself.

In [ ]:
manifest = repo / 'data' / 'tranches' / 'tahoe100m-pseudobulk-de.v1.manifest.txt'
tranche = json.loads((repo / 'data' / 'tranches' / 'tahoe100m-pseudobulk-de.v1.json').read_text())

manifest_sha = hashlib.sha256(manifest.read_bytes()).hexdigest()
print('manifest lines          :', len(manifest.read_text().splitlines()))
print('sha256(manifest)        :', manifest_sha)
print('tranche content_hash    :', tranche['content_hash'])
print('record data_commit      :', record['environment']['data_commit'])
assert len(manifest.read_text().splitlines()) == 1026
assert manifest_sha == tranche['content_hash'] == record['environment']['data_commit']

## 3. The headline's arithmetic holds together

The promoted table is one row. Display it whole, then check its internal arithmetic:
the Spearman-Brown full-data ceiling must equal 2r/(1+r) of the split-half mean, the
quartiles must bracket the median, both mismatch floors must sit below the observed
mean in the right order, the effect-size terciles must rise monotonically (the in-run
positive control), and both minimum detectable effects must sit below the observed
mean (the result is not a power artifact).

In [ ]:
headline = pd.read_csv(results / 'rung0_delta_reproducibility.csv').iloc[0]
print(headline.to_string())

m = headline['splithalf_mean_r']
print('\n2r/(1+r) =', round(2 * m / (1 + m), 3), '  promoted spearman_brown_full =', headline['spearman_brown_full'])
assert round(2 * m / (1 + m), 3) == headline['spearman_brown_full']
assert headline['splithalf_q1_r'] <= headline['splithalf_median_r'] <= headline['splithalf_q3_r']
assert headline['null_diff_drug_mean_r'] < headline['null_same_drug_mean_r'] < m
assert (headline['splithalf_mean_r_tercile1'] < headline['splithalf_mean_r_tercile2']
        < headline['splithalf_mean_r_tercile3'])
assert headline['mde_80_vs_diff_drug'] < m and headline['mde_80_vs_same_drug'] < m
print('internal arithmetic holds')

## 4. The 1,600 scored conditions are exactly the splittable ones

The run measured the composition of the pool it consumed. From that table: 1,650
candidate (cell line, drug) conditions; exactly the 50 Ribociclib rows have all their
replicate plates in one half (a single plate cannot be split); 1,650 − 50 must equal
the promoted pair count. One cell-line key is literally the string `NA` (a missing
DepMap id carried through from the source), so the read must keep it as text rather
than parse it as missing data.

In [ ]:
pool = pd.read_csv(task / 'rung0_pool_description.csv', keep_default_na=False)
unsplittable = pool[(pool['n_plates_half0'] == 0) | (pool['n_plates_half1'] == 0)]

print(len(pool), 'candidate conditions:', pool['patient'].nunique(), 'line keys x',
      pool['drug'].nunique(), 'drug names')
print(len(unsplittable), 'unsplittable, drug(s):', sorted(unsplittable['drug'].unique()))
print('scored =', len(pool) - len(unsplittable), '  promoted n_pairs =', int(headline['n_pairs']))
assert len(pool) == 1650 and pool['patient'].nunique() == 50 and pool['drug'].nunique() == 33
assert len(unsplittable) == 50 and set(unsplittable['drug']) == {'Ribociclib'}
assert len(pool) - len(unsplittable) == int(headline['n_pairs']) == 1600

## 5. The significance survives the dependence check

The reported p-values assume the mismatched-pair null draws behave like an exchangeable
pool although they reuse half-profiles. The task measured that assumption with shuffle
(derangement) checks — 500 permutations for the pooled comparison and for each of the
two comparison types the promotion reports, every per-permutation mean committed.
Recompute each null's mean and spread and the exact p = (1 + #{shuffles ≥ observed}) /
(1 + 500), and confirm the true pairing beats all 500 shuffles in every stratum.

In [ ]:
der = pd.read_csv(task / 'rung0_derangement_summary.csv').iloc[0]
strata = [
    ('any-pair', 'rung0_derangement_perm_means.csv', 'observed_mean', ''),
    ('same-drug', 'rung0_derangement_perm_means_same_drug.csv', 'observed_mean_same_drug_rows', '_same_drug'),
    ('diff-drug', 'rung0_derangement_perm_means_diff_drug.csv', 'observed_mean_diff_drug_rows', '_diff_drug'),
]
for name, filename, observed_col, suffix in strata:
    perms = pd.read_csv(task / filename)['perm_mean']
    observed = der[observed_col]
    p = (1 + int((perms >= observed).sum())) / (1 + len(perms))
    print(f'{name:9s}: {len(perms)} shuffles  mean {perms.mean():.4f}  sd {perms.std():.4f}  '
          f'max {perms.max():.4f} < observed {observed}  exact p {p:.3f}')
    assert len(perms) == 500 and observed > perms.max()
    assert round(perms.mean(), 4) == der['perm_mean_mean' + suffix]
    assert round(perms.std(), 4) == der['perm_mean_sd' + suffix]
    assert round(p, 3) == der['p_exact' + suffix]

The *design effect* is the shuffle-measured variance of the null mean divided by the
variance the exchangeable-pool shortcut assumed. Below one means the shortcut was
cautious, not generous. Recompute the pooled one from the committed draws (the
per-stratum ones need each stratum's pooled standard error, which only the cluster run
holds — they are transcribed in the summary and cross-checked in continuous
integration). Then confirm two entirely different sampling mechanisms — these
derangements and the headline's bootstrapped pools — land on the same mismatch floors.

In [ ]:
any_perms = pd.read_csv(task / 'rung0_derangement_perm_means.csv')['perm_mean']
design_effect = any_perms.var() / der['se_iid_pool'] ** 2
print('design effect =', round(design_effect, 3), '  reported:', der['design_effect'])
assert abs(design_effect - der['design_effect']) < 0.02 and design_effect < 1

print('same-drug floor: derangement', der['perm_mean_mean_same_drug'],
      ' bootstrap', headline['null_same_drug_mean_r'])
print('diff-drug floor: derangement', der['perm_mean_mean_diff_drug'],
      ' bootstrap', headline['null_diff_drug_mean_r'])
assert abs(der['perm_mean_mean_same_drug'] - headline['null_same_drug_mean_r']) < 0.0015
assert abs(der['perm_mean_mean_diff_drug'] - headline['null_diff_drug_mean_r']) < 0.0015

## 6. Reliability is broadly distributed, led by stress-response genes

The unpromoted per-gene diagnostic: 13,886 panel genes, each correlated across
conditions between the two plate halves. Recompute the write-up's numbers — 97.0%
positive, median 0.146, quartiles 0.089–0.230 — and look at the top of the table: it
should be heat-shock and immediate-early stress-response transcripts.

In [ ]:
pg = pd.read_csv(task / 'rung0_per_gene_reliability.csv')
finite = pg[np.isfinite(pg['r'])]

print(len(pg), 'genes,', len(finite), 'with a finite value,', len(pg) - len(finite), 'without')
print(f"positive fraction {100 * (finite['r'] > 0).mean():.1f}%  "
      f"median {finite['r'].median():.3f}  "
      f"quartiles {finite['r'].quantile(0.25):.3f}-{finite['r'].quantile(0.75):.3f}")
print(pg.nlargest(5, 'r').to_string(index=False))
assert len(pg) == 13886 and len(finite) == 13759
assert abs((finite['r'] > 0).mean() - 0.970) <= 0.0005
assert abs(finite['r'].median() - 0.146) <= 0.0005
assert abs(finite['r'].quantile(0.25) - 0.089) <= 0.0005
assert abs(finite['r'].quantile(0.75) - 0.230) <= 0.0005
assert list(pg.nlargest(5, 'r')['gene']) == ['HSP90AA1', 'EGR1', 'HSPA1B', 'HSPH1', 'PLEC']

## 7. The summary says what the artifacts say

Parse the evidence table out of `summary.md` and compare each number to the artifact it
came from — a transcribed number that drifts from its artifact fails right here. (The
prose paragraphs' numbers — terciles, power ratios, per-stratum design effects — get
the same treatment in the continuous-integration battery; section 8 runs it.)

In [ ]:
text = (task / 'summary.md').read_text()

def numbers(label):
    line = next(l for l in text.splitlines() if l.startswith('|') and label in l)
    return [float(x.replace(',', '')) for x in re.findall(r'\d[\d,]*\.?\d*', line.rsplit('|', 2)[-2])]

assert numbers('Conditions scored') == [headline['n_pairs']]
assert numbers('Panel genes present')[0] == headline['n_genes']  # 14,121 is hash-pinned, section 0
assert numbers('Split-half reliability') == [headline['splithalf_mean_r'], headline['splithalf_median_r'],
                                             headline['splithalf_q1_r'], headline['splithalf_q3_r']]
assert numbers('Spearman-Brown') == [headline['spearman_brown_full']]
assert numbers('positive reliability') == [round(100 * headline['frac_pos'], 1)]
assert numbers('Mismatched-condition floor')[-1] == headline['null_diff_drug_mean_r']
assert numbers('Same-drug floor')[-1] == headline['null_same_drug_mean_r']
assert numbers('Significance')[0] == headline['p_vs_null'] == headline['p_vs_same_drug']
assert numbers('Smallest detectable')[-2:] == [round(headline['mde_80_vs_diff_drug'], 3),
                                              round(headline['mde_80_vs_same_drug'], 3)]
print('every evidence-table number matches its artifact')

## 8. The instruments themselves are validated

Everything above checks the numbers; these two cells check the code that produced them.
First the known-answer suite: synthetic data with a planted reliability of 0.8 must
come out 0.8 through the real measurement, a signal-free pool must come out null, the
power calculation must match the closed-form normal-theory answer, and the one
statistical defect this project has actually shipped (comparing an aggregate against
single draws) is pinned by a test demonstrating the wrong form failing. Then the full
continuous-integration battery (`scripts/verify_rung0.py` — the scripted form of the
cells above plus the prose-paragraph checks), which must agree with everything you just
watched.

In [ ]:
result = subprocess.run(['uv', 'run', 'pytest', '-m', 'known_answer', '-q'],
                        cwd=repo, capture_output=True, text=True)
print(result.stdout[-2500:] + result.stderr[-500:])
assert result.returncode == 0, 'known-answer controls failed'

In [ ]:
battery = subprocess.run(['uv', 'run', 'python', 'scripts/verify_rung0.py'],
                         cwd=repo, capture_output=True, text=True)
print(battery.stdout[-600:])
assert battery.returncode == 0, 'the scripted battery disagrees with the cells above'
print('You have re-derived rung 0, not read about it.')